# Description

In this notebook I extract the NorESM2-LM outputs to use them with the TTD method and estimate the mean age. The model outputs required are temperature, salinity, cfc11, cfc12, sf6 as well as latitude, longitude, depth, year. Model output will be yearly averages and will be organize in a text file, each colum will be one of the variable, each row one data point.

Note that temperature and salinity will be read from the links in the shared directory, while cfcs and sf6 will be read in the NorESM2-LM outputs directory (/mnt/reef-ns1002k-ns9034k/CMIP6/CMIP/NCC/NorESM2-LM/historical/r1i1p1f1/). The first are not directly available in yearly averages so they have been preprocess in an other notebook; the second are in yearly average.


# Import modules

In [1]:
%%time
%load_ext memory_profiler

#___________________________
# basics
import datetime
import os, glob, sys, gc
# import warnings
# warnings.filterwarnings('ignore', '.*invalid value encountered in true_divide.*', )

#___________________________
# To follow computations
from dask.diagnostics import ProgressBar
pbar = ProgressBar(minimum=10)
pbar.register()
#pbar.unregister()

#___________________________
# xarray numpy...
import numpy as np
import xarray as xr
xr.set_options(keep_attrs=True)
import pandas as pd


CPU times: user 789 ms, sys: 1.15 s, total: 1.93 s
Wall time: 2.15 s


# Starters

In [2]:
%%time
%%memit -c
print('################################')
print('################################')
print(datetime.datetime.now())
print('################################')

dirout = '25-10-11-extract-noresm-outputs-for-TTD/'
if not os.path.isdir(dirout) : os.mkdir(dirout)

dirshared = '/mnt/reef-ns1002k/daco/MY_JUPYTER_NOTEBOOKS/OceanICU/SHARED-DATAS/'

netcdfdir = dirout+'netcdf_files/'
if not os.path.isdir(netcdfdir) : os.mkdir(netcdfdir)

sys.stdout.echo = open(dirout+'stdout.txt', 'w')
sys.stderr.echo = open(dirout+'stderr.txt', 'w')

################################
################################
2025-12-08 10:46:56.474797
################################
peak memory: 256.28 MiB, increment: 98.90 MiB
CPU times: user 27.6 ms, sys: 31.4 ms, total: 59 ms
Wall time: 178 ms


# Main parameters

In [3]:
%%time
%%memit -c
print('################################')
print('################################')
print(datetime.datetime.now())
print('################################')

kwopends=dict(use_cftime=True, decode_times=None,
              decode_cf=True, decode_coords=True)
kwopenmfds = dict(combine='by_coords', parallel=True, 
                  use_cftime=True, decode_times=None,
                  decode_cf=True, decode_coords=True)


rename_dict = {
    "x": "i",
    "y": "j",
    "lat": "latitude", 
    "lon": "longitude",
    "nav_lat": "latitude", 
    "nav_lon": "longitude",
    'lev': 'depth', 
    'deptht': 'depth', 
    'olevel': 'depth', 
    "Depth":"depth"
}


################################
################################
2025-12-08 10:46:56.666959
################################
peak memory: 256.32 MiB, increment: 107.32 MiB
CPU times: user 36.8 ms, sys: 13.4 ms, total: 50.2 ms
Wall time: 152 ms


# Define some functions

## Others

In [4]:
def check_and_delete_variable(variable_name):
    # Usage example
    # for vvv in ['data2plot', 'data2process_significant', 'data2process_with_low_significant']: 
    #     check_and_delete_variable(vvv)
    # #
    # gc.collect()    
    if variable_name in globals(): 
        del globals()[variable_name]
        print(f"{variable_name} deleted")
    elif variable_name in locals(): 
        del locals()[variable_name]
        print( f"{variable_name} deleted" )
    else:
        print( f"{variable_name} does not exist")

#

def get_esgf_dataset_filepaths(variable, sourceID, experimentID, 
                               freq='mon', grid='g*', version='latest', 
                               variant='r1i1p1f1',
                               mipera = 'CMIP6', diresgf='/mnt/reef-ns1002k-esgf/', verbose=False, **kwargs): 
    """
    Returns the filepaths of the remote netCDF files corresponding to the specified dataset of the Earth System
    Grid Federation (ESGF) data portal on NIRD.

    Parameters:
    -----------
    variable : str
        Variable to search for on ESGF data portal.
    sourceID : str
        Name of the data source on the ESGF data portal.
    experimentID : str
        Name of the experiment on the ESGF data portal.
    freq : str, optional
        Frequency of the data (default is 'mon').
    grid : str, optional
        Type of grid (default is 'g*').
    version : str, optional
        Version of the data being queried (default is 'latest').
    variant : str, optional
        Label for the variant of the data being queried (default is 'r1i1p1f1').
    mipera : str, optional
        Name of the CMIP era being queried (default is 'CMIP6').
    diresgf : str, optional
        Absolute path to the directory where the data is stored (default is '/mnt/reef-ns1002k-esgf/').
    verbose : bool, optional
        If True, prints the function name at the start and end of execution (default is False).
    **kwargs : dict, optional
        Other key-value arguments to be passed in the function.

    Returns:
    --------
    List[str]
        A list of filepaths corresponding to the specified dataset on the ESGF data portal.

    Example:
    --------
    fp_list = get_esgf_dataset_filepaths('tas', 'CanESM5', 'historical', freq='mon')

    Dependencies:
    -------------
    glob, sys
    """
    import glob, sys
    
    if verbose: print('func: get_esgf_dataset_filepaths')
    
    if experimentID in ['1pctCO2', 'piControl', 'historical', 'abrupt-4xCO2']: zwActivity='CMIP'
    elif experimentID in ['ssp126', 'ssp245', 'ssp585']: zwActivity='ScenarioMIP'
    else: sys.exit('Check experimentID, case not implemented')
    
    if sourceID in ['CESM2', 'CESM2-WACCM']: zwInstitutionID = 'NCAR'
    elif sourceID in ['ACCESS-ESM1-5']: zwInstitutionID = 'CSIRO'
    elif sourceID in ['CNRM-ESM2-1']: zwInstitutionID = 'CNRM-CERFACS'
    elif sourceID in ['CanESM5', 'CanESM5-CanOE']: zwInstitutionID = 'CCCma'
    elif sourceID in ['UKESM1-0-LL']: zwInstitutionID = 'NIMS-KMA'
    elif sourceID in ['GFDL-CM4', 'GFDL-ESM4']: zwInstitutionID = 'NOAA-GFDL'
    elif sourceID in ['IPSL-CM6A-LR', 'IPSL-CM6A-LR-INCA']: zwInstitutionID = 'IPSL'
    elif sourceID in ['MIROC-ES2L']: zwInstitutionID = 'MIROC'
    elif sourceID in ['MPI-ESM1-2-LR', 'ICON-ESM-LR']: zwInstitutionID = 'MPI-M'
    elif sourceID in ['NorESM2-LM']: zwInstitutionID = 'NCC'
    else: sys.exit('Check sourceID, case not implemented')
    
    ocean_list = ['fgco2', 'intpp', 'o2', 'thetao', 'so', 'agessc', 'po4', 'no3', 
                  'dissic', 'talk', 'cfc12', 'cfc11', 'sf6']
    if variable in ocean_list: zwTableID = 'O'+freq
    elif variable in ['areacello']: zwTableID='Ofx'
    elif variable in ['psl']: zwTableID='A'+freq
    else: sys.exit('!!! WARNING !!! Check variable, case not implemented')
        
    zwdname = diresgf + mipera +'/'+ zwActivity +'/'+ \
        zwInstitutionID +'/'+ sourceID +'/'+ \
        experimentID  +'/'+ variant +'/'+ zwTableID +'/'+ \
        variable+'/'+ grid +'/'+ version +'/'
    zwfname = variable +'_'+ zwTableID +'_'+ sourceID +'_'+ \
        experimentID +'_'+ variant +'_'+ grid +'*.nc' 

    if verbose: print('endfunc')
    return glob.glob(zwdname + zwfname)
#
def nan_helper(y):
    """Helper to handle indices and logical indices of NaNs.

    Input:
        - y, 1d numpy array with possible NaNs
    Output:
        - nans, logical indices of NaNs
        - index, a function, with signature indices= index(logical_indices),
          to convert logical indices of NaNs to 'equivalent' indices
    Example:
        >>> # linear interpolation of NaNs
        >>> nans, x= nan_helper(y)
        >>> y[nans]= np.interp(x(nans), x(~nans), y[~nans])
        nb: y[~nans] values of y that are not nans
            x(~nans) indexes of y that are not nans
    """

    return np.isnan(y), lambda z: z.nonzero()[0]
#



## Preparation of data 

In [5]:
def shift_180_lon(zwda, verbose=False): 
    if verbose: print("func: shift_180_lon")
    
    try: 
        if not np.nanmin(zwda['longitude']) < -150: 
            zwda['longitude'] = (zwda['longitude'] + 180) % 360 - 180
            addtxt=str(datetime.datetime.now())+' shift_180_lon to get longitude from -180 to 180'
            try: zwda.attrs['history'] =  addtxt + ' ; '+zwda.attrs['history']
            except: zwda.attrs['history'] =  addtxt             
        #
    except: print('WARNING! longitude likely not shifted')
    return zwda
#

def rename_vars_dims_coords(ds, rename_dict, verbose=False):
    """
    Renames variables, dimensions, and coordinates in an xarray Dataset according to the provided rename dictionary.

    Parameters:
    -----------
    ds : xr.Dataset
        The xarray Dataset to be renamed.
    rename_dict : Dict[str, str]
        Dictionary containing the variable, dimension, or coordinate names to be renamed. 
        The keys represent the original names, and the values represent the new names.
    verbose : bool, optional
        If True, prints the function name at the start and end of execution (default is False).
    
    Returns:
    --------
    xr.Dataset
        A new xarray Dataset with variables, dimensions, and coordinates renamed according to the rename dictionary.
    
    Example:
    --------
    import xarray as xr
    data = {'temp': ([], [0]), 'sali': ([], [1])}
    coords = {'time': [0]}
    ds = xr.Dataset(data, coords)
    renamed_ds = rename_vars_dims_coords(ds, {'temp': 'temperature', 'sali': 'salinity'})

    Dependencies:
    -------------
    xarray
    """
    if verbose: print('func: rename_vars_dims_coords')
    for old_name, new_name in rename_dict.items():
        if (old_name in ds.variables) | (old_name in ds.dims) | (old_name in ds.coords): 
            ds = ds.rename({old_name: new_name})
        #
    if verbose: print('endfunc')
    return ds
#

def split_coords_dimensions(ds, verbose=False):
    """
    Splits the latitude, longitude, and depth dimensions and coordinates of an xarray dataset into separate variables,
    updates their names, and assigns them back to the dataset.

    Parameters:
    -----------
    ds : xr.Dataset
        The xarray Dataset to be updated.
    verbose : bool, optional
        If True, prints the function name at the start and end of execution (default is False).
    
    Returns:
    --------
    xr.Dataset
        A new xarray Dataset with the latitude, longitude, and depth dimensions and coordinates split into separate variables
        and reassigned to the original dataset.
    
    Example:
    --------
    import xarray as xr
    data = {'temp': ([0, 1, 2], [0, 1]), 'sali': ([0, 1, 2], [0, 1])}
    coords = {'latitude': [0, 1, 2], 'longitude': [0, 1], 'depth': [0, 1, 2]}
    ds = xr.Dataset(data, coords)
    updated_ds = split_coords_dimensions(ds)

    Dependencies:
    -------------
    xarray
    """
    if verbose: print('func: split_coords_dimensions')
    new_coords = {}
    new_coords2 = {}
    new_dims = {}
    dim_name_dict = dict(latitude='j', longitude='i', depth='k')
    dimschanged = []
    for name, coord in ds.coords.items():
        if name in ds.dims and name in ["latitude", "longitude", "depth"]:
            new_coords[name + "_coord"] = coord
            new_dims[name] = dim_name_dict[name]
            new_coords2[name + "_coord"] = name
            dimschanged.append(name)
    if verbose: print('endfunc')
    for name in ['k', 'j', 'i']: 
        if name in ds.coords: dimschanged.append(name)
    #
    return ds.assign_coords(new_coords).rename_dims(new_dims).drop_vars(dimschanged).rename(new_coords2)
#


# Save global interior ocean

In [6]:
%%time
%%memit -c
# ca. 3 min
print('################################')
print('################################')
print(datetime.datetime.now())
print('# Save global interior ocean')
print('################################')

savename = 'NorESM2-LM_outputs_for_TTD_interior_ocean.txt'

#-----------------------
# Load data into dataarray
#-----------------------

print('Load data into dataarray...')

#________________
# Get potential temperature

print('    Get potential temperature...')

vesm = 'NorESM2-LM'
simu='historical'
var = 'thetao'
tslice = slice('1972', '2013')

fname = dirshared+vesm+"_"+simu+"_"+var+"_197[2-9].nc"
path_list = glob.glob(fname)
fname = dirshared+vesm+"_"+simu+"_"+var+"_19[89][0-9].nc"
path_list.extend(glob.glob(fname))
fname = dirshared+vesm+"_"+simu+"_"+var+"_200[0-9].nc"
path_list.extend(glob.glob(fname))
fname = dirshared+vesm+"_"+simu+"_"+var+"_201[0-3].nc"
path_list.extend(glob.glob(fname))
path_list.sort()

zwds = xr.open_mfdataset(path_list, **kwopenmfds) 
zwda_thetao = zwds[var].sel(year=tslice)


#________________
# Get practical salinity

print('    Get practical salinity...')

vesm = 'NorESM2-LM'
simu='historical'
var = 'so'
tslice = slice('1972', '2013')

fname = dirshared+vesm+"_"+simu+"_"+var+"_197[2-9].nc"
path_list = glob.glob(fname)
fname = dirshared+vesm+"_"+simu+"_"+var+"_19[89][0-9].nc"
path_list.extend(glob.glob(fname))
fname = dirshared+vesm+"_"+simu+"_"+var+"_200[0-9].nc"
path_list.extend(glob.glob(fname))
fname = dirshared+vesm+"_"+simu+"_"+var+"_201[0-3].nc"
path_list.extend(glob.glob(fname))
path_list.sort()

zwds = xr.open_mfdataset(path_list, **kwopenmfds) 
zwda_so = zwds[var].sel(year=tslice)

#________________
# Get CFC11

print('    Get CFC11...')

simu='historical'
esm  = 'NorESM2-LM' 
var = 'cfc11'
tslice = slice('1972', '2013')

fname_list = get_esgf_dataset_filepaths(var, esm, simu, 
                                        diresgf='/mnt/reef-ns1002k-ns9034k/', 
                                        version='v20191108', 
                                        variant='r1i1p1f1', grid='gr', freq='yr')
zwds = xr.open_mfdataset(fname_list, **kwopenmfds)
zwds2 = zwds[var].to_dataset()
zwds2 = rename_vars_dims_coords(zwds2, rename_dict)
zwds2 = split_coords_dimensions(zwds2)
zwds2 = shift_180_lon(zwds2)
zwda_cfc11 = zwds2[var].groupby('time.year').mean(dim='time').sel(year=tslice)

#________________
# Get CFC12

print('    Get CFC12...')

simu='historical'
esm  = 'NorESM2-LM' 
var = 'cfc12'
tslice = slice('1972', '2013')

fname_list = get_esgf_dataset_filepaths(var, esm, simu, 
                                        diresgf='/mnt/reef-ns1002k-ns9034k/', 
                                        version='v20191108', 
                                        variant='r1i1p1f1', grid='gr', freq='yr')
zwds = xr.open_mfdataset(fname_list, **kwopenmfds)
zwds2 = zwds[var].to_dataset()
zwds2 = rename_vars_dims_coords(zwds2, rename_dict)
zwds2 = split_coords_dimensions(zwds2)
zwds2 = shift_180_lon(zwds2)
zwda_cfc12 = zwds2[var].groupby('time.year').mean(dim='time').sel(year=tslice)

#________________
# Get SF6

print('    Get SF6...')

simu='historical'
esm  = 'NorESM2-LM' 
var = 'sf6'
tslice = slice('1972', '2013')

fname_list = get_esgf_dataset_filepaths(var, esm, simu, 
                                        diresgf='/mnt/reef-ns1002k-ns9034k/', 
                                        version='v20191108', 
                                        variant='r1i1p1f1', grid='gr', freq='yr')
zwds = xr.open_mfdataset(fname_list, **kwopenmfds)
zwds2 = zwds[var].to_dataset()
zwds2 = rename_vars_dims_coords(zwds2, rename_dict)
zwds2 = split_coords_dimensions(zwds2)
zwds2 = shift_180_lon(zwds2)
zwda_sf6 = zwds2[var].groupby('time.year').mean(dim='time').sel(year=tslice)

del zwds, zwds2
gc.collect()

#-----------------------
# Select interior ocean
#-----------------------

print('Select interior ocean...')

COND = (zwda_thetao['depth']>=1000)
zwda_thetao = zwda_thetao.where(COND).load()

COND = (zwda_so['depth']>=1000)
zwda_so = zwda_so.where(COND).load()

COND = (zwda_cfc11['depth']>=1000)
zwda_cfc11 = zwda_cfc11.where(COND).load()

COND = (zwda_cfc12['depth']>=1000)
zwda_cfc12 = zwda_cfc12.where(COND).load()

COND = (zwda_sf6['depth']>=1000)
zwda_sf6 = zwda_sf6.where(COND).load()

del COND
gc.collect()

#-----------------------
# Put data together and save in text file
#-----------------------

print('Put data together and save in text file...')

#________________
# Convert to pandas DataFrame, drop nans and combine
print('    Convert to pandas DataFrame, drop nans and combine...')

df_thetao = zwda_thetao.to_dataframe().reset_index()
del zwda_thetao
gc.collect()
df_so     = zwda_so.to_dataframe().reset_index()
del zwda_so
gc.collect()
df_cfc11  = zwda_cfc11.to_dataframe().reset_index()
del zwda_cfc11
gc.collect()
df_cfc12  = zwda_cfc12.to_dataframe().reset_index()
del zwda_cfc12
gc.collect()
df_sf6    = zwda_sf6.to_dataframe().reset_index()
del zwda_sf6
gc.collect()

df_thetao = df_thetao.dropna()
df_so     = df_so.dropna()
df_cfc11  = df_cfc11.dropna()
df_cfc12  = df_cfc12.dropna()
df_sf6    = df_sf6.dropna()

df_thetao = df_thetao[['year', 'depth', 'latitude', 'longitude', 'thetao']]
df_so     = df_so    [['year', 'depth', 'latitude', 'longitude', 'so'    ]]
df_cfc11  = df_cfc11 [['year', 'depth', 'latitude', 'longitude', 'cfc11' ]]
df_cfc12  = df_cfc12 [['year', 'depth', 'latitude', 'longitude', 'cfc12' ]]
df_sf6    = df_sf6   [['year', 'depth', 'latitude', 'longitude', 'sf6'   ]]

df_combined = pd.merge(df_thetao, df_so, on=['year', 'depth', 'latitude', 'longitude'])
del df_thetao, df_so
gc.collect()
df_combined = pd.merge(df_combined, df_cfc11, on=['year', 'depth', 'latitude', 'longitude'])
del df_cfc11
gc.collect()
df_combined = pd.merge(df_combined, df_cfc12, on=['year', 'depth', 'latitude', 'longitude'])
del df_cfc12
gc.collect()
df_combined = pd.merge(df_combined, df_sf6  , on=['year', 'depth', 'latitude', 'longitude'])
del df_sf6
gc.collect()

df_to_save = df_combined[['year', 'depth', 'latitude', 'longitude', 'thetao', 'so', 'cfc11', 'cfc12', 'sf6']]
del df_combined
gc.collect()


#________________
# Save into text file
print('    Save into text file...')

# Define the units for each column
units = {
    'year': 'years',
    'depth': 'meters',
    'latitude': 'degrees_north',
    'longitude': 'degrees_east',
    'thetao': 'Celsius',
    'so': 'PSU', 
    'cfc11': 'mol m-3', 
    'cfc12': 'mol m-3', 
    'sfs6' : 'mol m-3'
}

# Create header with column names and units
header = [f"{col} ({unit})" for col, unit in units.items()]

# Save the DataFrame to a text file with the custom header
df_to_save.to_csv(dirout+savename, sep='\t', index=False, header=header)

print('    File saved: '+dirout+savename)


print('################################')
print('Done: '+str(datetime.datetime.now()))
print('################################')
print('################################')

################################
################################
2025-11-17 11:17:02.039290
# Save global interior ocean
################################
Load data into dataarray...
    Get potential temperature...


Process MemTimer-3:
KeyboardInterrupt
Traceback (most recent call last):
  File "/opt/conda/lib/python3.10/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/home/daco/.local/lib/python3.10/site-packages/memory_profiler.py", line 262, in run
    stop = self.pipe.poll(self.interval)
  File "/opt/conda/lib/python3.10/multiprocessing/connection.py", line 262, in poll
    return self._poll(timeout)
  File "/opt/conda/lib/python3.10/multiprocessing/connection.py", line 429, in _poll
    r = wait([self], timeout)
  File "/opt/conda/lib/python3.10/multiprocessing/connection.py", line 936, in wait
    ready = selector.select(timeout)
  File "/opt/conda/lib/python3.10/selectors.py", line 416, in select
    fd_event_list = self._selector.poll(timeout)


KeyboardInterrupt: 

# Save small subset

In [17]:
%%time
%%memit -c
# ca. 3 min
print('################################')
print('################################')
print(datetime.datetime.now())
print('# Save small subset')
print('################################')

savename_suff = 'NorESM2-LM_outputs_for_TTD_small_subset'
tslice = slice('1990', '1995')

#-----------------------
# Load data into dataarray
#-----------------------

print('Load data into dataarray...')

#________________
# Get potential temperature

print('    Get potential temperature...')

vesm = 'NorESM2-LM'
simu='historical'
var = 'thetao'

fname = dirshared+vesm+"_"+simu+"_"+var+"_199[0-5].nc"
path_list = glob.glob(fname)
# fname = dirshared+vesm+"_"+simu+"_"+var+"_198[0-2].nc"
# path_list.extend(glob.glob(fname))
path_list.sort()

zwds = xr.open_mfdataset(path_list, **kwopenmfds) 
zwda_thetao = zwds[var].sel(year=tslice)


#________________
# Get practical salinity

print('    Get practical salinity...')

vesm = 'NorESM2-LM'
simu='historical'
var = 'so'

fname = dirshared+vesm+"_"+simu+"_"+var+"_199[0-5].nc"
path_list = glob.glob(fname)
# fname = dirshared+vesm+"_"+simu+"_"+var+"_198[0-2].nc"
# path_list.extend(glob.glob(fname))
path_list.sort()

zwds = xr.open_mfdataset(path_list, **kwopenmfds) 
zwda_so = zwds[var].sel(year=tslice)

# #________________
# # Get CFC11

# print('    Get CFC11...')

# simu='historical'
# esm  = 'NorESM2-LM' 
# var = 'cfc11'

# fname_list = get_esgf_dataset_filepaths(var, esm, simu, 
#                                         diresgf='/mnt/reef-ns1002k-ns9034k/', 
#                                         version='v20191108', 
#                                         variant='r1i1p1f1', grid='gr', freq='yr')
# zwds = xr.open_mfdataset(fname_list, **kwopenmfds)
# zwds2 = zwds[var].to_dataset()
# zwds2 = rename_vars_dims_coords(zwds2, rename_dict)
# zwds2 = split_coords_dimensions(zwds2)
# zwds2 = shift_180_lon(zwds2)
# zwda_cfc11 = zwds2[var].groupby('time.year').mean(dim='time').sel(year=tslice)

#________________
# Get CFC12

print('    Get CFC12...')

simu='historical'
esm  = 'NorESM2-LM' 
var = 'cfc12'

fname_list = get_esgf_dataset_filepaths(var, esm, simu, 
                                        diresgf='/mnt/reef-ns1002k-ns9034k/', 
                                        version='v20191108', 
                                        variant='r1i1p1f1', grid='gr', freq='yr')
zwds = xr.open_mfdataset(fname_list, **kwopenmfds)
zwds2 = zwds[var].to_dataset()
zwds2 = rename_vars_dims_coords(zwds2, rename_dict)
zwds2 = split_coords_dimensions(zwds2)
zwds2 = shift_180_lon(zwds2)
zwda_cfc12 = zwds2[var].groupby('time.year').mean(dim='time').sel(year=tslice)

#________________
# Get SF6

print('    Get SF6...')

simu='historical'
esm  = 'NorESM2-LM' 
var = 'sf6'

fname_list = get_esgf_dataset_filepaths(var, esm, simu, 
                                        diresgf='/mnt/reef-ns1002k-ns9034k/', 
                                        version='v20191108', 
                                        variant='r1i1p1f1', grid='gr', freq='yr')
zwds = xr.open_mfdataset(fname_list, **kwopenmfds)
zwds2 = zwds[var].to_dataset()
zwds2 = rename_vars_dims_coords(zwds2, rename_dict)
zwds2 = split_coords_dimensions(zwds2)
zwds2 = shift_180_lon(zwds2)
zwda_sf6 = zwds2[var].groupby('time.year').mean(dim='time').sel(year=tslice)

del zwds, zwds2
gc.collect()

#-----------------------
# Select interior ocean
#-----------------------

print('Select interior ocean...')

def define_condition(zwda): 
    zmin = 1000
    latmin, latmax = 40, 45
    lonmin, lonmax = -40, -35
    COND = (zwda['depth']>=zmin) & \
        (zwda['latitude']>=latmin)  & (zwda['latitude']<latmax) & \
        (zwda['longitude']>=lonmin)  & (zwda['longitude']<lonmax)
    return zwda.where(COND)
#

zwda_thetao = define_condition(zwda_thetao).load()
zwda_so = define_condition(zwda_so).load()
# zwda_cfc11 = define_condition(zwda_cfc11).load()
zwda_cfc12 = define_condition(zwda_cfc12).load()
zwda_sf6 = define_condition(zwda_sf6).load()


#-----------------------
# Put data together
#-----------------------

print('Put data together...')

#________________
# Convert to pandas DataFrame, drop nans and combine
print('    Convert to pandas DataFrame, drop nans and combine...')

df_thetao = zwda_thetao.to_dataframe().reset_index()
del zwda_thetao
gc.collect()
df_so     = zwda_so.to_dataframe().reset_index()
del zwda_so
gc.collect()
# df_cfc11  = zwda_cfc11.to_dataframe().reset_index()
# del zwda_cfc11
# gc.collect()
df_cfc12  = zwda_cfc12.to_dataframe().reset_index()
del zwda_cfc12
gc.collect()
df_sf6    = zwda_sf6.to_dataframe().reset_index()
del zwda_sf6
gc.collect()

df_thetao = df_thetao.dropna()
df_so     = df_so.dropna()
# df_cfc11  = df_cfc11.dropna()
df_cfc12  = df_cfc12.dropna()
df_sf6    = df_sf6.dropna()

df_thetao = df_thetao[['year', 'depth', 'latitude', 'longitude', 'thetao']]
df_so     = df_so    [['year', 'depth', 'latitude', 'longitude', 'so'    ]]
# df_cfc11  = df_cfc11 [['year', 'depth', 'latitude', 'longitude', 'cfc11' ]]
df_cfc12  = df_cfc12 [['year', 'depth', 'latitude', 'longitude', 'cfc12' ]]
df_sf6    = df_sf6   [['year', 'depth', 'latitude', 'longitude', 'sf6'   ]]

df_combined = pd.merge(df_thetao, df_so, on=['year', 'depth', 'latitude', 'longitude'])
del df_thetao, df_so
gc.collect()
# df_combined = pd.merge(df_combined, df_cfc11, on=['year', 'depth', 'latitude', 'longitude'])
# del df_cfc11
# gc.collect()
df_combined = pd.merge(df_combined, df_cfc12, on=['year', 'depth', 'latitude', 'longitude'])
del df_cfc12
gc.collect()
df_combined = pd.merge(df_combined, df_sf6  , on=['year', 'depth', 'latitude', 'longitude'])
del df_sf6
gc.collect()

df_to_save_compact_cfc12 = df_combined[['year', 'thetao', 'so', 'cfc12']]
df_to_save_compact_sf6 = df_combined[['year', 'thetao', 'so', 'sf6']]
df_to_save_extended = df_combined[['year', 'depth', 'latitude', 'longitude', 'thetao', 'so', 'cfc12', 'sf6']]
del df_combined
gc.collect()


#-----------------------
# Save into text file
#-----------------------

print('Save into text file...')

#_________________
# COMPACT CFC12
print('    Save dataframe compact_cfc12...')

# Define the units for each column
units = {
    'year': 'years',
    'thetao': 'Celsius',
    'so': 'PSU', 
    'cfc12': 'mol m-3', 
}
# Create header with column names and units
header = [f"{col} ({unit})" for col, unit in units.items()]
# Save the DataFrame to a text file with the custom header
savename = dirout+savename_suff+'_compact_cfc12.txt'
df_to_save_compact_cfc12.to_csv(savename, sep='\t', index=False, header=header)

print('    File saved: '+savename)
print(f'    Number of rows in file: {df_to_save.shape[0]}')

#_________________
# COMPACT SF6
print('    Save dataframe compact_sf6...')

# Define the units for each column
units = {
    'year': 'years',
    'thetao': 'Celsius',
    'so': 'PSU', 
    'sf6': 'mol m-3', 
}
# Create header with column names and units
header = [f"{col} ({unit})" for col, unit in units.items()]
# Save the DataFrame to a text file with the custom header
savename = dirout+savename_suff+'_compact_sf6.txt'
df_to_save_compact_sf6.to_csv(savename, sep='\t', index=False, header=header)

print('    File saved: '+savename)
print(f'    Number of rows in file: {df_to_save.shape[0]}')

#_________________
# Extended
print('    Save dataframe extended...')

# Define the units for each column
units = {
    'year': 'years',
    'depth': 'meters',
    'latitude': 'degrees_north',
    'longitude': 'degrees_east',
    'thetao': 'Celsius',
    'so': 'PSU', 
    'cfc12': 'mol m-3', 
    'sfs6' : 'mol m-3'
}
# Create header with column names and units
header = [f"{col} ({unit})" for col, unit in units.items()]
# Save the DataFrame to a text file with the custom header
savename = dirout+savename_suff+'_extended.txt'
df_to_save_extended.to_csv(savename, sep='\t', index=False, header=header)

print('    File saved: '+savename)
print(f'    Number of rows in file: {df_to_save.shape[0]}')


print('################################')
print('Done: '+str(datetime.datetime.now()))
print('################################')
print('################################')

# Save all time period, all ocean cut in parts

In [6]:
%%time
%%memit -c
# ca. 3 min
print('################################')
print('################################')
print(datetime.datetime.now())
print('# Save all time period, all ocean cut in parts')
print('################################')

savename_suff = 'NorESM2-LM_outputs_for_TTD'
tslice = slice('1972', '2013')

#-----------------------
# Load data into dataarray
#-----------------------

print('Load data into dataarray...')

#________________
# Get potential temperature

print('    Get potential temperature...')

vesm = 'NorESM2-LM'
simu='historical'
var = 'thetao'

fname = dirshared+vesm+"_"+simu+"_"+var+"_197[2-9].nc"
path_list = glob.glob(fname)
fname = dirshared+vesm+"_"+simu+"_"+var+"_19[89][0-9].nc"
path_list.extend(glob.glob(fname))
fname = dirshared+vesm+"_"+simu+"_"+var+"_200[0-9].nc"
path_list.extend(glob.glob(fname))
fname = dirshared+vesm+"_"+simu+"_"+var+"_201[0-3].nc"
path_list.extend(glob.glob(fname))
path_list.sort()

zwds = xr.open_mfdataset(path_list, **kwopenmfds) 
zwda_thetao = zwds[var].sel(year=tslice)


#________________
# Get practical salinity

print('    Get practical salinity...')

vesm = 'NorESM2-LM'
simu='historical'
var = 'so'

fname = dirshared+vesm+"_"+simu+"_"+var+"_197[2-9].nc"
path_list = glob.glob(fname)
fname = dirshared+vesm+"_"+simu+"_"+var+"_19[89][0-9].nc"
path_list.extend(glob.glob(fname))
fname = dirshared+vesm+"_"+simu+"_"+var+"_200[0-9].nc"
path_list.extend(glob.glob(fname))
fname = dirshared+vesm+"_"+simu+"_"+var+"_201[0-3].nc"
path_list.extend(glob.glob(fname))
path_list.sort()

zwds = xr.open_mfdataset(path_list, **kwopenmfds) 
zwda_so = zwds[var].sel(year=tslice)

#________________
# Get CFC12

print('    Get CFC12...')

simu='historical'
esm  = 'NorESM2-LM' 
var = 'cfc12'

fname_list = get_esgf_dataset_filepaths(var, esm, simu, 
                                        diresgf='/mnt/reef-ns1002k-ns9034k/', 
                                        version='v20191108', 
                                        variant='r1i1p1f1', grid='gr', freq='yr')
zwds = xr.open_mfdataset(fname_list, **kwopenmfds)
zwds2 = zwds[var].to_dataset()
zwds2 = rename_vars_dims_coords(zwds2, rename_dict)
zwds2 = split_coords_dimensions(zwds2)
zwds2 = shift_180_lon(zwds2)
zwda_cfc12 = zwds2[var].groupby('time.year').mean(dim='time').sel(year=tslice)

#________________
# Get SF6

print('    Get SF6...')

simu='historical'
esm  = 'NorESM2-LM' 
var = 'sf6'

fname_list = get_esgf_dataset_filepaths(var, esm, simu, 
                                        diresgf='/mnt/reef-ns1002k-ns9034k/', 
                                        version='v20191108', 
                                        variant='r1i1p1f1', grid='gr', freq='yr')
zwds = xr.open_mfdataset(fname_list, **kwopenmfds)
zwds2 = zwds[var].to_dataset()
zwds2 = rename_vars_dims_coords(zwds2, rename_dict)
zwds2 = split_coords_dimensions(zwds2)
zwds2 = shift_180_lon(zwds2)
zwda_sf6 = zwds2[var].groupby('time.year').mean(dim='time').sel(year=tslice)

del zwds, zwds2
gc.collect()

#________________
# Get region mask

print('    Get region mask...')

file_regions= '/mnt/reef-ns1002k/daco/ocean_regions_tnx1v4_20190729.nc'
zwds_regions = xr.open_dataset(file_regions, **kwopends)
zwds2 = zwds_regions['region'].to_dataset()
zwds2 = rename_vars_dims_coords(zwds2, rename_dict)
zwds2 = split_coords_dimensions(zwds2)
# zwds2 = shift_180_lon(zwds2)
region = zwds2['region']

region_names = zwds_regions['region_names']
region_dict = {
    'southern': 1, 
    'atlantic': 2, 
    'pacific': 3, 
    'indian': 5
}


#-----------------------
# Select ocean region, convert to dataframe and drop nans
#-----------------------

######
######
######
zwregion = 'atlantic'
latbnds = [    
    (60, 80)
]
# latbnds = [    
#     (-80, -70), (-70, -65), (-65, -60), (-60, -50)
# ]
# latbnds = [    
#     (-60, -50), (-50, -40), (-40, -30), (-30, -20), (-20, -10), (-10, 0), \
#     (0, 10), (10, 20), (20, 30), (30, 40), (40, 50), (50, 60)
# ]
######
######
######

for latmin, latmax in latbnds: 
    
    if latmin>=0: lll1=f'{latmin}N'
    else: lll1=f'{-latmin}S'
    if latmax>=0: lll2=f'{latmax}N'
    else: lll2=f'{-latmax}S'
    savename_add = f'{zwregion}_{lll1}-{lll2}'
    
    print(f'For region {savename_add}...')

    print('    Select ocean region, convert to dataframe and drop nans...')
    
    def select_convert_drop(zwda, zwda_region, zwregionnum, zwlatmin, zwlatmax): 
        zmin = 1000
        COND = (zwda['depth']>=zmin) & (zwda_region==zwregionnum) & \
            (zwda['latitude']>=zwlatmin)  & (zwda['latitude']<zwlatmax)
        zwda_sel = zwda.where(COND)
        zwdf = zwda_sel.to_dataframe().reset_index()
        zwdf = zwdf.dropna()
        return zwdf
    #


    print('        For thetao...')
    df_thetao = select_convert_drop(zwda_thetao, region, region_dict[zwregion], latmin, latmax)

    print('        For so...')
    df_so = select_convert_drop(zwda_so, region, region_dict[zwregion], latmin, latmax)

    print('        For cfc12...')
    df_cfc12 = select_convert_drop(zwda_cfc12, region, region_dict[zwregion], latmin, latmax)

    print('        For sf6...')
    df_sf6 = select_convert_drop(zwda_sf6, region, region_dict[zwregion], latmin, latmax)


    #-----------------------
    # Put data together
    #-----------------------

    print('    Put data together...')
    df_thetao = df_thetao[['year', 'depth', 'latitude', 'longitude', 'thetao']]
    df_so     = df_so    [['year', 'depth', 'latitude', 'longitude', 'so'    ]]
    df_cfc12  = df_cfc12 [['year', 'depth', 'latitude', 'longitude', 'cfc12' ]]
    df_sf6    = df_sf6   [['year', 'depth', 'latitude', 'longitude', 'sf6'   ]]

    df_combined = pd.merge(df_thetao, df_so, on=['year', 'depth', 'latitude', 'longitude'])
    del df_thetao, df_so
    gc.collect()
    df_combined = pd.merge(df_combined, df_cfc12, on=['year', 'depth', 'latitude', 'longitude'])
    del df_cfc12
    gc.collect()
    df_combined = pd.merge(df_combined, df_sf6  , on=['year', 'depth', 'latitude', 'longitude'])
    del df_sf6
    gc.collect()

    df_to_save_compact_cfc12 = df_combined[['year', 'thetao', 'so', 'cfc12']]
    df_to_save_compact_sf6 = df_combined[['year', 'thetao', 'so', 'sf6']]
    df_to_save_extended = df_combined[['year', 'depth', 'latitude', 'longitude', 'thetao', 'so', 'cfc12', 'sf6']]
    del df_combined
    gc.collect()


    #-----------------------
    # Save into text file
    #-----------------------

    print('    Save into text file...')

    #_________________
    # COMPACT CFC12
    print('        Save dataframe compact_cfc12...')

    # Define the units for each column
    units = {
        'year': 'years',
        'thetao': 'Celsius',
        'so': 'PSU', 
        'cfc12': 'mol m-3', 
    }
    # Create header with column names and units
    header = [f"{col} ({unit})" for col, unit in units.items()]
    # Save the DataFrame to a text file with the custom header
    savename = f'{dirout}{savename_suff}_{savename_add}_compact_cfc12.txt'
    df_to_save_compact_cfc12.to_csv(savename, sep='\t', index=False, header=header)

    print('        File saved: '+savename)
    print(f'        Number of rows in file: {df_to_save_compact_cfc12.shape[0]}')

    #_________________
    # COMPACT SF6
    print('        Save dataframe compact_sf6...')

    # Define the units for each column
    units = {
        'year': 'years',
        'thetao': 'Celsius',
        'so': 'PSU', 
        'sf6': 'mol m-3', 
    }
    # Create header with column names and units
    header = [f"{col} ({unit})" for col, unit in units.items()]
    # Save the DataFrame to a text file with the custom header
    savename = f'{dirout}{savename_suff}_{savename_add}_compact_sf6.txt'
    df_to_save_compact_sf6.to_csv(savename, sep='\t', index=False, header=header)

    print('        File saved: '+savename)
    print(f'        Number of rows in file: {df_to_save_compact_sf6.shape[0]}')


    #_________________
    # Extended
    print('        Save dataframe extended...')

    # Define the units for each column
    units = {
        'year': 'years',
        'depth': 'meters',
        'latitude': 'degrees_north',
        'longitude': 'degrees_east',
        'thetao': 'Celsius',
        'so': 'PSU', 
        'cfc12': 'mol m-3', 
        'sfs6' : 'mol m-3'
    }
    # Create header with column names and units
    header = [f"{col} ({unit})" for col, unit in units.items()]
    # Save the DataFrame to a text file with the custom header
    savename = f'{dirout}{savename_suff}_{savename_add}_extended.txt'
    df_to_save_extended.to_csv(savename, sep='\t', index=False, header=header)

    print('        File saved: '+savename)
    print(f'        Number of rows in file: {df_to_save_extended.shape[0]}')

#

print('################################')
print('Done: '+str(datetime.datetime.now()))
print('################################')
print('################################')

################################
################################
2025-12-08 10:47:21.165647
# Save all time period, all ocean cut in parts
################################
Load data into dataarray...
    Get potential temperature...
    Get practical salinity...
    Get CFC12...
    Get SF6...
    Get region mask...
For region atlantic_60N-80N...
    Select ocean region, convert to dataframe and drop nans...
        For thetao...
        For so...
        For cfc12...
        For sf6...
    Put data together...
    Save into text file...
        Save dataframe compact_cfc12...
        File saved: 25-10-11-extract-noresm-outputs-for-TTD/NorESM2-LM_outputs_for_TTD_atlantic_60N-80N_compact_cfc12.txt
        Number of rows in file: 651420
        Save dataframe compact_sf6...
        File saved: 25-10-11-extract-noresm-outputs-for-TTD/NorESM2-LM_outputs_for_TTD_atlantic_60N-80N_compact_sf6.txt
        Number of rows in file: 651420
        Save dataframe extended...
        File saved: 25

# Save all time period, all ocean cut in parts with dimensions

In [7]:
%%time
%%memit -c
# ca. 3 min
print('################################')
print('################################')
print(datetime.datetime.now())
print('# Save all time period, all ocean cut in parts with dimensions')
print('################################')

savename_suff = 'NorESM2-LM_outputs_for_TTD'
tslice = slice('1972', '2013')

#-----------------------
# Load data into dataarray
#-----------------------

print('Load data into dataarray...')

#________________
# Get potential temperature

print('    Get potential temperature...')

vesm = 'NorESM2-LM'
simu='historical'
var = 'thetao'

fname = dirshared+vesm+"_"+simu+"_"+var+"_197[2-9].nc"
path_list = glob.glob(fname)
fname = dirshared+vesm+"_"+simu+"_"+var+"_19[89][0-9].nc"
path_list.extend(glob.glob(fname))
fname = dirshared+vesm+"_"+simu+"_"+var+"_200[0-9].nc"
path_list.extend(glob.glob(fname))
fname = dirshared+vesm+"_"+simu+"_"+var+"_201[0-3].nc"
path_list.extend(glob.glob(fname))
path_list.sort()

zwds = xr.open_mfdataset(path_list, **kwopenmfds) 
zwda_thetao = zwds[var].sel(year=tslice)


#________________
# Get practical salinity

print('    Get practical salinity...')

vesm = 'NorESM2-LM'
simu='historical'
var = 'so'

fname = dirshared+vesm+"_"+simu+"_"+var+"_197[2-9].nc"
path_list = glob.glob(fname)
fname = dirshared+vesm+"_"+simu+"_"+var+"_19[89][0-9].nc"
path_list.extend(glob.glob(fname))
fname = dirshared+vesm+"_"+simu+"_"+var+"_200[0-9].nc"
path_list.extend(glob.glob(fname))
fname = dirshared+vesm+"_"+simu+"_"+var+"_201[0-3].nc"
path_list.extend(glob.glob(fname))
path_list.sort()

zwds = xr.open_mfdataset(path_list, **kwopenmfds) 
zwda_so = zwds[var].sel(year=tslice)

#________________
# Get CFC12

print('    Get CFC12...')

simu='historical'
esm  = 'NorESM2-LM' 
var = 'cfc12'

fname_list = get_esgf_dataset_filepaths(var, esm, simu, 
                                        diresgf='/mnt/reef-ns1002k-ns9034k/', 
                                        version='v20191108', 
                                        variant='r1i1p1f1', grid='gr', freq='yr')
zwds = xr.open_mfdataset(fname_list, **kwopenmfds)
zwds2 = zwds[var].to_dataset()
zwds2 = rename_vars_dims_coords(zwds2, rename_dict)
zwds2 = split_coords_dimensions(zwds2)
zwds2 = shift_180_lon(zwds2)
zwda_cfc12 = zwds2[var].groupby('time.year').mean(dim='time').sel(year=tslice)

#________________
# Get SF6

print('    Get SF6...')

simu='historical'
esm  = 'NorESM2-LM' 
var = 'sf6'

fname_list = get_esgf_dataset_filepaths(var, esm, simu, 
                                        diresgf='/mnt/reef-ns1002k-ns9034k/', 
                                        version='v20191108', 
                                        variant='r1i1p1f1', grid='gr', freq='yr')
zwds = xr.open_mfdataset(fname_list, **kwopenmfds)
zwds2 = zwds[var].to_dataset()
zwds2 = rename_vars_dims_coords(zwds2, rename_dict)
zwds2 = split_coords_dimensions(zwds2)
zwds2 = shift_180_lon(zwds2)
zwda_sf6 = zwds2[var].groupby('time.year').mean(dim='time').sel(year=tslice)

del zwds, zwds2
gc.collect()

#________________
# Get region mask

print('    Get region mask...')

file_regions= '/mnt/reef-ns1002k/daco/ocean_regions_tnx1v4_20190729.nc'
zwds_regions = xr.open_dataset(file_regions, **kwopends)
zwds2 = zwds_regions['region'].to_dataset()
zwds2 = rename_vars_dims_coords(zwds2, rename_dict)
zwds2 = split_coords_dimensions(zwds2)
# zwds2 = shift_180_lon(zwds2)
region = zwds2['region']

region_names = zwds_regions['region_names']
region_dict = {
    'southern': 1, 
    'atlantic': 2, 
    'pacific': 3, 
    'indian': 5
}


#-----------------------
# Select ocean region, convert to dataframe and drop nans
#-----------------------

######
######
######
zwregion = 'atlantic'
latbnds = [    
    (60, 80)
]
# latbnds = [    
#     (-80, -70), (-70, -65), (-65, -60), (-60, -50)
# ]
# latbnds = [    
#     (-60, -50), (-50, -40), (-40, -30), (-30, -20), (-20, -10), (-10, 0), \
#     (0, 30)
# ]
# latbnds = [    
#     (-60, -50), (-50, -40), (-40, -30), (-30, -20), (-20, -10), (-10, 0), \
#     (0, 10), (10, 20), (20, 30), (30, 40), (40, 50), (50, 60)
# ]
######
######
######

for latmin, latmax in latbnds: 
    
    if latmin>=0: lll1=f'{latmin}N'
    else: lll1=f'{-latmin}S'
    if latmax>=0: lll2=f'{latmax}N'
    else: lll2=f'{-latmax}S'
    savename_add = f'{zwregion}_{lll1}-{lll2}'
    
    print(f'For region {savename_add}...')

    print('    Select ocean region, convert to dataframe and drop nans...')
    
    def select_convert_drop(zwda, zwda_region, zwregionnum, zwlatmin, zwlatmax): 
        zmin = 1000
        COND = (zwda['depth']>=zmin) & (zwda_region==zwregionnum) & \
            (zwda['latitude']>=zwlatmin)  & (zwda['latitude']<zwlatmax)
        zwda_sel = zwda.where(COND)
        zwdf = zwda_sel.to_dataframe().reset_index()
        zwdf = zwdf.dropna()
        return zwdf
    #


    print('        For thetao...')
    df_thetao = select_convert_drop(zwda_thetao, region, region_dict[zwregion], latmin, latmax)

    print('        For so...')
    df_so = select_convert_drop(zwda_so, region, region_dict[zwregion], latmin, latmax)

    print('        For cfc12...')
    df_cfc12 = select_convert_drop(zwda_cfc12, region, region_dict[zwregion], latmin, latmax)

    print('        For sf6...')
    df_sf6 = select_convert_drop(zwda_sf6, region, region_dict[zwregion], latmin, latmax)

    #-----------------------
    # Put data together
    #-----------------------

    print('    Put data together...')

    df_combined = pd.merge(df_thetao, df_so, on=['year', 'depth', 'latitude', 'longitude'], suffixes=("", "_so"))
    del df_thetao, df_so
    gc.collect()
    df_combined = pd.merge(df_combined, df_cfc12, on=['year', 'depth', 'latitude', 'longitude'], suffixes=("", "_cfc12"))
    del df_cfc12
    gc.collect()
    df_combined = pd.merge(df_combined, df_sf6  , on=['year', 'depth', 'latitude', 'longitude'], suffixes=("", "_sf6"))
    del df_sf6
    gc.collect()

    df_to_save_extended = df_combined[['year', 'depth', 'latitude', 'longitude', 'k', 'j', 'i', 'thetao', 'so', 'cfc12', 'sf6']]
    del df_combined
    gc.collect()


    #-----------------------
    # Save into text file
    #-----------------------

    print('    Save into text file...')

    # Define the units for each column
    units = {
        'year': 'years',
        'depth': 'meters',
        'latitude': 'degrees_north',
        'longitude': 'degrees_east',
        'k': '', 
        'j': '', 
        'i': '',
        'thetao': 'Celsius',
        'so': 'PSU', 
        'cfc12': 'mol m-3', 
        'sfs6' : 'mol m-3'
    }
    # Create header with column names and units
    header = [f"{col} ({unit})" for col, unit in units.items()]
    # Save the DataFrame to a text file with the custom header
    savename = f'{dirout}{savename_suff}_{savename_add}_extended_with_dimensions.txt'
    df_to_save_extended.to_csv(savename, sep='\t', index=False, header=header)

    print('        File saved: '+savename)
    print(f'        Number of rows in file: {df_to_save_extended.shape[0]}')

#

print('################################')
print('Done: '+str(datetime.datetime.now()))
print('################################')
print('################################')

################################
################################
2025-12-08 10:54:14.353243
# Save all time period, all ocean cut in parts with dimensions
################################
Load data into dataarray...
    Get potential temperature...
    Get practical salinity...
    Get CFC12...
    Get SF6...
    Get region mask...
For region atlantic_60N-80N...
    Select ocean region, convert to dataframe and drop nans...
        For thetao...
        For so...
        For cfc12...
        For sf6...
    Put data together...
    Save into text file...
        File saved: 25-10-11-extract-noresm-outputs-for-TTD/NorESM2-LM_outputs_for_TTD_atlantic_60N-80N_extended_with_dimensions.txt
        Number of rows in file: 651420
################################
Done: 2025-12-08 10:56:41.088336
################################
################################
peak memory: 53247.10 MiB, increment: 50840.86 MiB
CPU times: user 1min 22s, sys: 42.1 s, total: 2min 4s
Wall time: 2min 26s
